# RLVR with a Custom Reward Function

## Lab 2 - Create and register the reward function

This notebook tests `reward_function.py` locally and registers it as a SageMaker AI Registry Evaluator with type `REWARD_FUNCTION`.

## 1. Set up SageMaker

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

project_prefix = "gsm8k-custom-reward-rlvr"
# Keep the evaluator name short because SageMaker creates an underlying Lambda
# function named SageMaker-evaluator-<name>-<timestamp>, and Lambda function
# names must be <= 64 characters.
reward_function_name = "gsm8k-rlvr-rf"
print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")

## 2. Smoke test the reward function locally

The SageMaker runtime sends a batch list. The last assistant message is the model response being scored.

In [ ]:
import json
from reward_function import lambda_handler

sample_event = [
    {
        "id": "gsm8k-smoke-000001",
        "messages": [
            {"role": "user", "content": "What is 6 times 7? Think step by step and output the final answer after ####."},
            {"role": "assistant", "content": "6 times 7 is 42, so the final answer is #### 42"},
        ],
        "reference_answer": "42",
    },
    {
        "id": "gsm8k-smoke-000002",
        "messages": [
            {"role": "user", "content": "What is 10 plus 5?"},
            {"role": "assistant", "content": "The answer is #### 12"},
        ],
        "reference_answer": "15",
    },
]

response = lambda_handler(sample_event, None)
print(json.dumps(response, indent=2))
results = json.loads(response["body"])
assert response["statusCode"] == 200
assert results[0]["aggregate_reward_score"] == 1.0
assert results[1]["aggregate_reward_score"] < 1.0
print("Smoke test passed.")

## 3. Register the reward function evaluator

SageMaker AI Registry supports Reward Function evaluators from either a Lambda ARN or bring-your-own code. This lab uses bring-your-own code by passing the local `reward_function.py` path.

In [ ]:
from sagemaker.ai_registry.air_constants import REWARD_FUNCTION
from sagemaker.ai_registry.evaluator import Evaluator

reward_function_evaluator = Evaluator.create(
    name=reward_function_name,
    type=REWARD_FUNCTION,
    source="reward_function.py",
    role=role,
    sagemaker_session=sess,
    wait=True,
)

reward_function_evaluator.refresh()
print(f"Reward function ARN: {reward_function_evaluator.arn}")